# Comprehensive Intrinsic Metrics Evaluation
## exp2b_flash_learned_pool — Validation Set Analysis

**Purpose:** Evaluate trained model on validation set without retraining.
Compute micro/macro recall and precision @10/@20, plus breakdowns by code type.

**Model:** `exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt`

### Metrics Computed
1. **Global metrics** — Micro/Macro Recall@10, @20; Micro/Macro Precision@10, @20; NDCG; MRR; Positive Brier
2. **Code-type metrics** — All above broken down by: ICD-10, Procedures, GPI, Provider, Revenue, DRG, Days, Place of Service

### Table of Contents
1. Environment Setup
2. Configuration
3. Load Trained Model
4. Load Validation Data
5. Load w2ind_target & Build Code Type Mapping
6. Comprehensive Evaluation
7. Global Metrics Results
8. Code-Type-Specific Results
9. Visualization
10. Save Results


In [ ]:
import sys
import os
import gc
import time
import json
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Any, Set
from dataclasses import dataclass, field
from collections import defaultdict

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.cloud import bigquery
from IPython.display import display

warnings.filterwarnings("ignore")

MODULE_DIR = os.getcwd()
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

from moe_flashattn_4_core import (
    BaseConfig,
    FlashAttentionConfig,
    MoEConfig,
    ClinicalDatasetLazy,
    create_collate_fn,
    FlashAttentionTransformer,
    FlashMoETransformer,
    BaselineTransformer,
    DataParallelWrapper,
    StreamingMetrics,
    prepare_data_once,
    cleanup_gpu_memory,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {props.total_memory / 1e9:.1f} GB")


## 2. Configuration

### Key Parameters
- Model checkpoint path on GCP Vertex
- Training data table (to reconstruct validation split)
- w2ind_target table (for code type classification)
- K values for top-K metrics
- Train/val split ratio and seed (MUST match training run)


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

TRAINED_MODEL_PATH = (
    "logs/exp_round10_3lobs_formal_training/"
    "exp2b_flash_learned_pool_formal/saved_models/"
    "exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt"
)

TRAINING_DATA_TABLE = (
    "edp-prod-storage.edp_ent_sdoheir_cns."
    "a834793_Combined_All_LOB_o3_train_ending"
)

W2IND_TARGET_TABLE = (
    "edp-prod-storage.edp_ent_sdoheir_cns."
    "a834793_member_w2ind_target"
)

# CRITICAL: Must match the training run exactly
TRAIN_RATIO = 0.99    # 99% train / 1% validation (formal training)
RANDOM_SEED = 42

# Top-K values for metrics
K_VALUES = (1, 5, 10, 20, 50)
PRIMARY_K_VALUES = (10, 20)  # For detailed reporting

# Evaluation
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 4

# Macro metrics: minimum sample count per code for inclusion
MACRO_MIN_COUNT = 5

print(f"Model: {TRAINED_MODEL_PATH}")
print(f"Training data: {TRAINING_DATA_TABLE}")
print(f"w2ind_target: {W2IND_TARGET_TABLE}")
print(f"Train/Val split: {TRAIN_RATIO}/{1-TRAIN_RATIO:.2f} (seed={RANDOM_SEED})")
print(f"K values: {K_VALUES}")
print(f"Primary K values for detailed report: {PRIMARY_K_VALUES}")
assert os.path.exists(TRAINED_MODEL_PATH), f"Model not found: {TRAINED_MODEL_PATH}"


## 3. Load Trained Model

Load the trained FlashAttentionTransformer from checkpoint.
The checkpoint contains model_state_dict, config, and model_type.


In [ ]:
# ============================================================================
# MODEL LOADING FROM CHECKPOINT
# ============================================================================
# Source: dev/downstream/moe_flashattn_3_lob3_downstream_running.ipynb
#
# Reconstructs the full model architecture from checkpoint metadata and loads
# the saved weights.  Handles all three model families:
#   - BaselineTransformer
#   - FlashAttentionTransformer
#   - FlashMoETransformer (with automatic d_ff inference from expert weights)

def load_model_from_checkpoint(
    model_path: str,
    device: torch.device,
    verbose: bool = True,
) -> Tuple[torch.nn.Module, BaseConfig, Optional[MoEConfig], bool, str]:
    """
    Load a pretrained model from a .pt checkpoint.

    Checkpoint expected keys:
        model_state_dict, model_type, config, moe_config (optional)

    Returns:
        (model, config, moe_config, use_mixed_precision, model_type)
    """
    if verbose:
        print(f"\n{'=' * 70}")
        print(f"Loading model from: {model_path}")

    checkpoint_data = torch.load(model_path, map_location=device, weights_only=False)

    model_type = checkpoint_data.get("model_type", "Unknown")
    config_dict = checkpoint_data.get("config", {})
    moe_config_dict = checkpoint_data.get("moe_config", None)
    state_dict = checkpoint_data["model_state_dict"]

    if verbose:
        print(f"  Model type: {model_type}")
        print(f"  Embedding size: {config_dict.get('embedding_size', 256)}")
        print(f"  N layers: {config_dict.get('nlayers', 6)}")
        print(f"  Learned attention pooling: {config_dict.get('use_learnt_att_pool', False)}")

    # Infer learned pooling from state_dict keys (ground truth)
    use_learnt_att_pool_inferred = "daily_pooling.query" in state_dict

    # For MoE models, infer d_ff from expert weight shapes to avoid mismatch
    inferred_d_ff = None
    if "FlashMoE" in model_type:
        for key in state_dict.keys():
            if "experts.0.ffn.w_gate.weight" in key:
                d_ff_adjusted = state_dict[key].shape[0]
                inferred_d_ff = (d_ff_adjusted * 3 + 1) // 2
                if verbose:
                    print(f"  Inferred d_ff from expert weights: {inferred_d_ff}")
                break
        if inferred_d_ff is None:
            inferred_d_ff = config_dict.get("nhid", 512)

    # For non-MoE FlashAttention models, infer nhid from dense FFN weight shapes
    inferred_nhid = None
    if "FlashMoE" not in model_type:
        for key in state_dict.keys():
            if "temporal_layers.0.ffn.w_gate.weight" in key:
                d_ff_adjusted = state_dict[key].shape[0]
                inferred_nhid = (d_ff_adjusted * 3 + 1) // 2
                if verbose:
                    print(f"  Inferred nhid from FFN weights: {inferred_nhid} "
                          f"(d_ff_adjusted={d_ff_adjusted})")
                break

    # --- Reconstruct model by type ---
    moe_config_out = None

    if "FlashMoE" in model_type:
        config = FlashAttentionConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nhead=config_dict.get("nhead", 8),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get("use_swiglu", True),
            use_rope=config_dict.get("use_rope", True),
            use_flash=config_dict.get("use_flash", True),
        )
        d_ff_to_use = inferred_d_ff or config_dict.get("nhid", 512)
        if moe_config_dict:
            if verbose and moe_config_dict.get("d_ff") != d_ff_to_use:
                print(f"  Correcting d_ff: checkpoint={moe_config_dict.get('d_ff')}, actual={d_ff_to_use}")
            moe_config_out = MoEConfig(
                d_model=moe_config_dict.get("d_model", config.embedding_size),
                d_ff=d_ff_to_use,
                num_experts=moe_config_dict.get("num_experts", 8),
                num_shared_experts=moe_config_dict.get("num_shared_experts", 1),
                top_k=moe_config_dict.get("top_k", 2),
                expert_dropout=moe_config_dict.get("expert_dropout", 0.1),
                load_balance_strategy=moe_config_dict.get("load_balance_strategy", "deepseek"),
                aux_loss_weight=moe_config_dict.get("aux_loss_weight", 0.001),
                use_moe_from_layer=moe_config_dict.get("use_moe_from_layer", 2),
                use_swiglu_experts=moe_config_dict.get("use_swiglu_experts", True),
                router_warmup_steps=moe_config_dict.get("router_warmup_steps", 0),
                z_loss_weight=moe_config_dict.get("z_loss_weight", 0.005),
                bias_lr=moe_config_dict.get("bias_lr", 1e-3),
                bias_momentum=moe_config_dict.get("bias_momentum", 0.6),
            )
        else:
            moe_config_out = MoEConfig(d_model=config.embedding_size, d_ff=config.nhid)
        model = FlashMoETransformer(config, moe_config_out)
        use_mixed_precision = True

    elif "FlashAttention" in model_type:
        nhid_to_use = inferred_nhid or config_dict.get("nhid", 512)
        if verbose and config_dict.get("nhid") and inferred_nhid and config_dict["nhid"] != inferred_nhid:
            print(f"  Warning: config nhid={config_dict['nhid']} vs inferred nhid={inferred_nhid}, "
                  f"using weight-inferred value")
        config = FlashAttentionConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=nhid_to_use,
            nhead=config_dict.get("nhead", 8),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get("use_swiglu", True),
            use_rope=config_dict.get("use_rope", True),
            use_flash=config_dict.get("use_flash", True),
        )
        model = FlashAttentionTransformer(config)
        use_mixed_precision = True

    else:
        config = BaseConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
        )
        model = BaselineTransformer(config)
        use_mixed_precision = False

    model.load_state_dict(checkpoint_data["model_state_dict"])
    model = model.to(device)
    model.eval()

    if verbose:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  Model loaded successfully!")
        print(f"  Total parameters: {total_params:,}")
        print(f"  Mixed precision: {use_mixed_precision}")
        print(f"  Device: {device}")
        print(f"{'=' * 70}\n")

    return model, config, moe_config_out, use_mixed_precision, model_type

In [ ]:
cleanup_gpu_memory(verbose=False)

model, config, moe_config_loaded, use_mixed_precision, model_type = load_model_from_checkpoint(
    model_path=TRAINED_MODEL_PATH,
    device=device,
    verbose=True,
)

print(f"\nConfig summary:")
print(f"  target_cd_cnt: {config.target_cd_cnt}")
print(f"  len_dy: {config.len_dy}")
print(f"  len_cd: {config.len_cd}")
print(f"  cd_cnt: {config.cd_cnt}")
print(f"  embedding_size: {config.embedding_size}")


## 4. Load Validation Data

Reconstruct the exact validation split used during training:
1. Load full training table from BigQuery
2. Deduplicate (single-record members only)
3. Stratified split with same TRAIN_RATIO and RANDOM_SEED
4. Keep only val_df; free training data from memory


In [ ]:
client = bigquery.Client()

training_sql = f"""
SELECT *
FROM `{TRAINING_DATA_TABLE}`
"""

print("Loading training data from BigQuery (full dataset)...")
print(f"Table: {TRAINING_DATA_TABLE}")
start_time = time.time()

input_data = client.query(training_sql).to_dataframe()

elapsed = time.time() - start_time
print(f"Loaded {len(input_data):,} rows in {elapsed:.1f}s")
print(f"Columns: {list(input_data.columns)}")
print(f"Memory: {input_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")


In [ ]:
# Deduplicate: keep only members with exactly 1 record
member_counts = input_data.groupby("individual_id").size()
single_record_members = member_counts[member_counts == 1].index
df_unique = input_data[input_data["individual_id"].isin(single_record_members)].copy()

validation_provenance = {
    "training_data_table": TRAINING_DATA_TABLE,
    "loaded_training_rows": int(len(input_data)),
    "single_record_member_pool": int(len(df_unique)),
    "train_ratio": float(TRAIN_RATIO),
    "validation_ratio_nominal": float(1 - TRAIN_RATIO),
    "random_seed": int(RANDOM_SEED),
}

del input_data
gc.collect()

print(f"Unique members (single record): {len(df_unique):,}")
print(f"LOB distribution:\n{df_unique['lob'].value_counts()}")

# Stratified train/val split — MUST match training run
train_df, val_df = train_test_split(
    df_unique,
    train_size=TRAIN_RATIO,
    stratify=df_unique["lob"],
    random_state=RANDOM_SEED,
)

validation_provenance["validation_members"] = int(len(val_df))
validation_provenance["validation_unique_individual_ids"] = int(
    val_df["individual_id"].nunique()
    if "individual_id" in val_df.columns else len(val_df)
)
validation_provenance["validation_member_share_of_pool"] = float(
    len(val_df) / max(len(single_record_members), 1)
)
validation_provenance["validation_lob_distribution"] = {
    str(key): int(value) for key, value in val_df["lob"].value_counts().to_dict().items()
}

# Free training data immediately — we only need val_df
del train_df, df_unique
gc.collect()

print(f"\nValidation set: {len(val_df):,} members")
print(f"Val LOB distribution:\n{val_df['lob'].value_counts()}")

In [ ]:
val_dataset = ClinicalDatasetLazy(val_df, config)
val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=create_collate_fn(config),
    pin_memory=True,
    drop_last=False,
)

validation_provenance["val_dataset_rows"] = int(len(val_dataset))
validation_provenance["val_loader_batches"] = int(len(val_loader))

print(f"Validation DataLoader: {len(val_loader)} batches "
      f"({len(val_dataset)} samples, batch_size={EVAL_BATCH_SIZE})")
print("Validation provenance summary:")
print(f"  Loaded training rows: {validation_provenance['loaded_training_rows']:,}")
print(f"  Single-record member pool: {validation_provenance['single_record_member_pool']:,}")
print(f"  Validation members: {validation_provenance['validation_members']:,}")
print(f"  Validation share of pool: {validation_provenance['validation_member_share_of_pool']:.4f}")

## 5. Code Type Classification

Load w2ind_target to map each target vocabulary index to its code string,
then classify by code type using prefix patterns.

Code types (from `create_w2ind_target_from_w2ind.sql`):
| Type | Prefix | Example |
|------|--------|---------|
| ICD-10 Diagnosis | `icd9_dx_cd` | icd9_dx_cdG24 |
| Procedure Groups | `prcdr_group_` | prcdr_group_992 |
| GPI Medications | `gpi` | gpi22 |
| Provider Taxonomy | `provider_taxonomy_cd` | provider_taxonomy_cd207Q |
| Revenue Codes | `revenue_cd` | revenue_cd025 |
| DRG Codes | `drg_cd` | drg_cd470 |
| Days Count | `days_cnt` | days_cnt_5 |
| Place of Service | `hcfa_plc_srv_cd` | hcfa_plc_srv_cd21 |


In [ ]:
w2ind_target_sql = f"""
SELECT cd, ind
FROM `{W2IND_TARGET_TABLE}`
WHERE cd IS NOT NULL AND cd != ''
ORDER BY ind
"""

print("Loading w2ind_target from BigQuery...")
w2ind_target_df = client.query(w2ind_target_sql).to_dataframe()
print(f"Loaded {len(w2ind_target_df):,} target codes")

# Build index → code string mapping
idx_to_code = {}
for _, row in w2ind_target_df.iterrows():
    idx_to_code[int(row['ind'])] = str(row['cd'])

print(f"Index range: 1 to {max(idx_to_code.keys())}")
print(f"Sample mappings: {dict(list(idx_to_code.items())[:5])}")


In [ ]:
def classify_code_type(code_str: str) -> str:
    """
    Classify a target code string into its code type category.
    Mirrors the grouping logic in create_w2ind_target_from_w2ind.sql.
    """
    if code_str.startswith('icd9_dx_cd'):
        return 'ICD-10 Diagnosis'
    elif code_str.startswith('prcdr_group_'):
        return 'Procedure Groups'
    elif code_str.startswith('gpi'):
        return 'GPI Medications'
    elif code_str.startswith('provider_taxonomy_cd'):
        return 'Provider Taxonomy'
    elif code_str.startswith('revenue_cd'):
        return 'Revenue Codes'
    elif code_str.startswith('drg_cd'):
        return 'DRG Codes'
    elif code_str.startswith('days_cnt'):
        return 'Days Count'
    elif code_str.startswith('hcfa_plc_srv_cd'):
        return 'Place of Service'
    else:
        return 'Other'

# Build index → code_type mapping
idx_to_type = {}
for idx, code_str in idx_to_code.items():
    idx_to_type[idx] = classify_code_type(code_str)

# Build code_type → set of indices mapping
type_to_indices = defaultdict(set)
for idx, code_type in idx_to_type.items():
    type_to_indices[code_type].add(idx)

# Summary
print("Code Type Distribution in Target Vocabulary:")
print("-" * 50)
for code_type in sorted(type_to_indices.keys()):
    indices = type_to_indices[code_type]
    print(f"  {code_type:25s}: {len(indices):,} codes")
print(f"  {'TOTAL':25s}: {len(idx_to_type):,} codes")

# Create a numpy array for fast lookup: idx → type_id
CODE_TYPES = sorted(type_to_indices.keys())
type_to_id = {t: i for i, t in enumerate(CODE_TYPES)}
idx_type_array = np.full(config.target_cd_cnt, -1, dtype=np.int32)
for idx, code_type in idx_to_type.items():
    if idx < config.target_cd_cnt:
        idx_type_array[idx] = type_to_id[code_type]

print(f"\nType ID mapping: {type_to_id}")


## 6. Comprehensive Evaluation

### Approach
1. Reuse `StreamingMetrics` for sample-level metrics (recall@K, precision@K, NDCG, MRR, Brier)
2. Add per-code accumulators (numpy arrays, ~126KB total) for macro metrics:
   - `code_true_count[c]`: times code c appears in ground truth
   - `code_topk_count[k][c]`: times code c appears in top-K predictions
   - `code_hit_count[k][c]`: times code c appears in BOTH ground truth AND top-K
3. After evaluation, derive macro and code-type metrics from per-code arrays


In [ ]:
class PerCodeAccumulator:
    """
    Memory-efficient per-code tracking for macro metrics and code-type breakdowns.

    Maintains 3 types of counters as numpy arrays (vocab_size ≈ 6297):
    - code_true_count: ground truth frequency per code
    - code_topk_count: top-K prediction frequency per code (one array per K)
    - code_hit_count: hit frequency per code (one array per K)

    Additionally tracks member-level (sample-level) counts per code type:
    - type_member_true_count[type_id]: samples with >=1 true code of this type
    - type_member_hit_count[k][type_id]: samples with >=1 hit of this type in top-K

    Total memory: ~5 arrays × 6297 × 8 bytes + ~2 × 8 types × 5 K × 8 bytes ≈ 253KB
    """

    def __init__(
        self,
        vocab_size: int,
        k_values: Tuple[int, ...],
        idx_type_array: np.ndarray = None,
        num_types: int = 0,
    ):
        self.vocab_size = vocab_size
        self.k_values = k_values
        self._max_k = max(k_values)

        self.code_true_count = np.zeros(vocab_size, dtype=np.int64)
        self.code_topk_count = {k: np.zeros(vocab_size, dtype=np.int64) for k in k_values}
        self.code_hit_count = {k: np.zeros(vocab_size, dtype=np.int64) for k in k_values}
        self.total_samples = 0

        # Member-level tracking per code type (all on CPU to avoid GPU pressure)
        self._idx_type_array = idx_type_array
        self._num_types = num_types
        if idx_type_array is not None and num_types > 0:
            # Pre-build per-type boolean masks on CPU [num_types, vocab_size]
            self._type_masks_cpu = np.stack([
                (idx_type_array == tid) for tid in range(num_types)
            ])  # shape: [num_types, vocab_size]
            self.type_member_true_count = np.zeros(num_types, dtype=np.int64)
            self.type_member_hit_count = {k: np.zeros(num_types, dtype=np.int64) for k in k_values}
            self.overall_member_hit_count = {k: 0 for k in k_values}
        else:
            self._type_masks_cpu = None
            self.type_member_true_count = None
            self.type_member_hit_count = None
            self.overall_member_hit_count = None

    def update(
        self,
        predictions: torch.Tensor,   # [batch, vocab_size] logits
        targets: List[List[int]],     # target code lists per sample
    ) -> None:
        """Update per-code counters from a batch.

        GPU: only the original code-level aggregation (identical to pre-change).
        CPU: member-level per-type counting uses targets list + top-K index
        arrays directly — zero additional GPU tensors or transfers.
        """
        batch_size = predictions.shape[0]
        device = predictions.device

        # Build target boolean tensor [batch, vocab_size]
        target_tensor = torch.zeros(
            batch_size, self.vocab_size, dtype=torch.bool, device=device
        )
        valid_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Also collect per-sample valid code sets for CPU member tracking
        valid_code_sets = []

        for i, target_codes in enumerate(targets):
            valid_codes = [c for c in target_codes if 0 < c < self.vocab_size]
            if valid_codes:
                target_tensor[i, valid_codes] = True
                valid_mask[i] = True
                valid_code_sets.append(set(valid_codes))
            else:
                valid_code_sets.append(None)

        num_valid = valid_mask.sum().item()
        if num_valid == 0:
            return

        self.total_samples += num_valid

        # Update code_true_count: sum targets over batch
        code_true_batch = target_tensor[valid_mask].sum(dim=0).cpu().numpy()
        self.code_true_count += code_true_batch

        # Get top-K indices (compute once for max K)
        with torch.no_grad():
            _, top_k_indices = torch.topk(predictions, self._max_k, dim=-1)

        # CPU: get top-K indices as numpy for member-level tracking
        topk_np = None
        if self._type_masks_cpu is not None:
            topk_np = top_k_indices.cpu().numpy()  # [batch, max_k] — small transfer

        for k in self.k_values:
            top_k = top_k_indices[:, :k]  # [batch, k]

            # Build top-K boolean tensor [batch, vocab_size]
            topk_tensor = torch.zeros(
                batch_size, self.vocab_size, dtype=torch.bool, device=device
            )
            topk_tensor.scatter_(1, top_k, True)

            # code_topk_count: how many valid samples have this code in top-K
            topk_valid = topk_tensor[valid_mask].sum(dim=0).cpu().numpy()
            self.code_topk_count[k] += topk_valid

            # code_hit_count: intersection of targets and top-K
            hits = (target_tensor & topk_tensor)[valid_mask].sum(dim=0).cpu().numpy()
            self.code_hit_count[k] += hits

            del topk_tensor

        # Cleanup GPU tensors
        del target_tensor, top_k_indices

        # CPU-only: member-level per-type tracking using code sets + topk indices
        if self._type_masks_cpu is not None and topk_np is not None:
            idx_types = self._idx_type_array  # [vocab_size] → type_id
            for i, code_set in enumerate(valid_code_sets):
                if code_set is None:
                    continue

                # Which types does this sample have true codes for?
                sample_true_types = set()
                for c in code_set:
                    tid = idx_types[c]
                    if tid >= 0:
                        sample_true_types.add(tid)

                for tid in sample_true_types:
                    self.type_member_true_count[tid] += 1

                # Per-K: check which types have hits
                for k in self.k_values:
                    topk_codes = topk_np[i, :k]
                    hit_codes = code_set.intersection(topk_codes)
                    if hit_codes:
                        self.overall_member_hit_count[k] += 1
                        hit_types = set()
                        for c in hit_codes:
                            tid = idx_types[c]
                            if tid >= 0:
                                hit_types.add(tid)
                        for tid in hit_types:
                            self.type_member_hit_count[k][tid] += 1

    def compute_macro_metrics(
        self, min_count: int = 5
    ) -> Dict[str, float]:
        """
        Compute macro-averaged recall and precision at each K.

        Args:
            min_count: Minimum ground-truth count for a code to be included
                       in macro averaging (avoids noise from ultra-rare codes).
        """
        metrics = {}

        for k in self.k_values:
            # Macro Recall@K
            active_mask = self.code_true_count >= min_count
            if active_mask.sum() > 0:
                per_code_recall = np.divide(
                    self.code_hit_count[k],
                    self.code_true_count,
                    out=np.zeros_like(self.code_hit_count[k], dtype=np.float64),
                    where=active_mask,
                )
                macro_recall = per_code_recall[active_mask].mean()
                metrics[f'macro_recall@{k}'] = float(macro_recall)
                metrics[f'macro_recall@{k}_num_codes'] = int(active_mask.sum())
            else:
                metrics[f'macro_recall@{k}'] = 0.0
                metrics[f'macro_recall@{k}_num_codes'] = 0

            # Macro Precision@K
            pred_mask = self.code_topk_count[k] >= min_count
            if pred_mask.sum() > 0:
                per_code_precision = np.divide(
                    self.code_hit_count[k],
                    self.code_topk_count[k],
                    out=np.zeros_like(self.code_hit_count[k], dtype=np.float64),
                    where=pred_mask,
                )
                macro_precision = per_code_precision[pred_mask].mean()
                metrics[f'macro_precision@{k}'] = float(macro_precision)
                metrics[f'macro_precision@{k}_num_codes'] = int(pred_mask.sum())
            else:
                metrics[f'macro_precision@{k}'] = 0.0
                metrics[f'macro_precision@{k}_num_codes'] = 0

        return metrics

    def compute_code_type_metrics(
        self,
        idx_type_array: np.ndarray,
        type_to_id: Dict[str, int],
        code_types: List[str],
        min_count: int = 1,
    ) -> Dict[str, Dict[str, float]]:
        """
        Compute metrics broken down by code type.

        Returns:
            Dict[code_type_name → Dict[metric_name → value]]
        """
        results = {}

        for code_type in code_types:
            type_id = type_to_id[code_type]
            type_mask = idx_type_array == type_id
            num_codes_in_type = type_mask.sum()

            if num_codes_in_type == 0:
                continue

            type_metrics = {
                'num_codes': int(num_codes_in_type),
                'total_true_occurrences': int(self.code_true_count[type_mask].sum()),
            }

            # Member-level counts for this code type
            if self.type_member_true_count is not None:
                type_metrics['members_with_true_codes'] = int(
                    self.type_member_true_count[type_id]
                )

            for k in self.k_values:
                type_true = self.code_true_count[type_mask]
                type_topk = self.code_topk_count[k][type_mask]
                type_hits = self.code_hit_count[k][type_mask]

                # Micro Recall: total hits of this type / total true of this type
                total_true = type_true.sum()
                total_hits = type_hits.sum()
                total_topk = type_topk.sum()

                type_metrics[f'micro_recall@{k}'] = (
                    float(total_hits / total_true) if total_true > 0 else 0.0
                )
                type_metrics[f'micro_precision@{k}'] = (
                    float(total_hits / total_topk) if total_topk > 0 else 0.0
                )

                # Member-level hit count at this K for this code type
                if self.type_member_hit_count is not None:
                    type_metrics[f'members_identified@{k}'] = int(
                        self.type_member_hit_count[k][type_id]
                    )

                # Macro Recall: average per-code recall within this type
                active = type_true >= min_count
                if active.sum() > 0:
                    per_code_recall = np.divide(
                        type_hits, type_true,
                        out=np.zeros_like(type_hits, dtype=np.float64),
                        where=active,
                    )
                    type_metrics[f'macro_recall@{k}'] = float(per_code_recall[active].mean())
                    type_metrics[f'macro_recall@{k}_num_codes'] = int(active.sum())
                else:
                    type_metrics[f'macro_recall@{k}'] = 0.0
                    type_metrics[f'macro_recall@{k}_num_codes'] = 0

                # Macro Precision: average per-code precision within this type
                pred_active = type_topk >= min_count
                if pred_active.sum() > 0:
                    per_code_prec = np.divide(
                        type_hits, type_topk,
                        out=np.zeros_like(type_hits, dtype=np.float64),
                        where=pred_active,
                    )
                    type_metrics[f'macro_precision@{k}'] = float(per_code_prec[pred_active].mean())
                    type_metrics[f'macro_precision@{k}_num_codes'] = int(pred_active.sum())
                else:
                    type_metrics[f'macro_precision@{k}'] = 0.0
                    type_metrics[f'macro_precision@{k}_num_codes'] = 0

            results[code_type] = type_metrics

        return results

from contextlib import nullcontext

def _model_has_moe(model):
    """Check if model has MoE layers."""
    if isinstance(model, nn.DataParallel):
        model = model.module
    if isinstance(model, DataParallelWrapper):
        model = model.model
    return hasattr(model, 'temporal_layers') and any(
        hasattr(layer, 'moe') for layer in model.temporal_layers
    )

def evaluate_comprehensive(
    model: nn.Module,
    dataloader: DataLoader,
    config: BaseConfig,
    device: torch.device,
    k_values: Tuple[int, ...] = (1, 5, 10, 20, 50),
    use_mixed_precision: bool = True,
    idx_type_array: np.ndarray = None,
    type_to_id: Dict[str, int] = None,
    code_types: List[str] = None,
    macro_min_count: int = 5,
) -> Dict[str, Any]:
    """
    Run comprehensive evaluation with sample-level, macro, and code-type metrics.

    Returns dict with keys:
        'sample_level': StreamingMetrics results
        'macro': Macro recall/precision results
        'code_type': Per-code-type breakdown
        'per_code': Raw per-code accumulators (for further analysis)
    """
    model.eval()
    num_batches = len(dataloader)

    is_wrapped = isinstance(model, DataParallelWrapper) or (
        isinstance(model, nn.DataParallel) and
        isinstance(model.module, DataParallelWrapper)
    )

    # Standard sample-level streaming metrics
    streaming = StreamingMetrics(
        k_values=k_values,
        compute_mrr=True,
        compute_brier=True,
        vocab_size=config.target_cd_cnt,
    )

    # Per-code accumulator for macro metrics (with member-level type tracking)
    per_code = PerCodeAccumulator(
        vocab_size=config.target_cd_cnt,
        k_values=k_values,
        idx_type_array=idx_type_array,
        num_types=len(code_types) if code_types else 0,
    )

    # Loss criterion (for val_loss computation)
    criterion = nn.BCEWithLogitsLoss()

    print(f"Evaluating {num_batches} batches...")
    eval_start = time.time()

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Evaluating")):
            # === Forward pass ===
            age = batch['age'].to(device, non_blocking=True)
            gender = batch['gender'].to(device, non_blocking=True)
            lob = batch['lob'].to(device, non_blocking=True)
            codes = batch['codes'].to(device, non_blocking=True)
            dt_cnt = batch['dt_cnt']
            y = batch['target']

            x = torch.cat([
                age.unsqueeze(-1),
                gender.unsqueeze(-1),
                lob.unsqueeze(-1),
                codes,
            ], dim=-1)

            autocast_ctx = (
                torch.cuda.amp.autocast(dtype=torch.float16)
                if use_mixed_precision else nullcontext()
            )

            with autocast_ctx:
                if is_wrapped:
                    targets_mh = batch['target_multihot'].to(device, non_blocking=True)
                    dt_cnt_tensor = (
                        dt_cnt.to(device) if isinstance(dt_cnt, torch.Tensor)
                        else torch.tensor(dt_cnt, device=device)
                    )
                    result = model(x, dt_cnt_tensor, targets_mh, return_predictions=True)
                    if isinstance(result, tuple):
                        loss_val, extras = result
                        output = extras.get('predictions')
                    else:
                        loss_val = result
                        output = None
                    loss = loss_val.mean().item() if loss_val.numel() > 1 else loss_val.item()
                else:
                    if _model_has_moe(model):
                        output, _ = model(x, return_moe_losses=False)
                    else:
                        output = model(x)
                    # Loss computed after valid-day flattening (compute_loss not in moe_flashattn_4_core)
                    loss = None

            if output is None:
                streaming.update_loss(loss)
                continue

            # === Flatten over valid days ===
            batch_size = output.shape[0]
            actual_len_dy = output.shape[1]

            dt_cnt_values = dt_cnt.cpu().tolist() if isinstance(dt_cnt, torch.Tensor) else dt_cnt
            y_flat = [item for sublist in y for item in sublist]

            output_flat = output.view(batch_size * actual_len_dy, config.target_cd_cnt)

            valid_outputs = []
            valid_targets = []

            for j in range(batch_size):
                valid_days = min(int(dt_cnt_values[j]), actual_len_dy)
                if valid_days <= 0:
                    continue

                out_start = actual_len_dy * j
                out_end = out_start + valid_days
                valid_outputs.append(output_flat[out_start:out_end])

                y_start = config.len_dy * j
                y_end = y_start + valid_days
                valid_targets.extend(y_flat[y_start:y_end])

            if valid_outputs:
                predictions_flat = torch.cat(valid_outputs)

                # Deferred BCE loss for non-wrapped models (matches comprehensive_evaluation)
                if loss is None:
                    from moe_flashattn_4_core import create_multihot_targets_vectorized
                    targets_multihot = create_multihot_targets_vectorized(
                        valid_targets, len(predictions_flat), config.target_cd_cnt, device
                    )
                    loss = criterion(predictions_flat, targets_multihot).item()

                # Update sample-level metrics
                streaming.update_loss(loss)
                streaming.update(predictions_flat, valid_targets)

                # Update per-code metrics
                per_code.update(predictions_flat, valid_targets)

                del predictions_flat, valid_outputs, valid_targets

            del output
            if batch_idx % 500 == 0:
                gc.collect()

    eval_time = time.time() - eval_start
    print(f"\nEvaluation completed in {eval_time:.1f}s ({eval_time/60:.1f} min)")

    # === Compute all metrics ===
    sample_metrics = streaming.compute()
    macro_metrics = per_code.compute_macro_metrics(min_count=macro_min_count)

    code_type_metrics = {}
    if idx_type_array is not None and type_to_id is not None and code_types is not None:
        code_type_metrics = per_code.compute_code_type_metrics(
            idx_type_array, type_to_id, code_types, min_count=1,
        )

    # Cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return {
        'sample_level': sample_metrics,
        'macro': macro_metrics,
        'code_type': code_type_metrics,
        'per_code': per_code,
        'eval_time_sec': eval_time,
    }


In [ ]:
results = evaluate_comprehensive(
    model=model,
    dataloader=val_loader,
    config=config,
    device=device,
    k_values=K_VALUES,
    use_mixed_precision=use_mixed_precision,
    idx_type_array=idx_type_array,
    type_to_id=type_to_id,
    code_types=CODE_TYPES,
    macro_min_count=MACRO_MIN_COUNT,
)

print("Evaluation complete!")
print(f"  Total valid (sample, day) pairs evaluated: {results['sample_level'].get('num_samples', 'N/A'):,}")
print(f"  Evaluation time: {results['eval_time_sec']:.1f}s")


## 7. Global Metrics Results

### 7.1 Sample-Level Metrics (from StreamingMetrics)


In [ ]:
sl = results['sample_level']

print("=" * 70)
print("SAMPLE-LEVEL METRICS")
print("=" * 70)
print(f"\n  Val Loss:        {sl.get('val_loss', 0):.6f}")
print(f"  MRR:             {sl.get('mrr', 0):.4f}")
print(f"  Positive Brier:  {sl.get('positive_brier', 0):.4f}")

print(f"\n  {'K':>4s}  {'Recall@K':>10s}  {'MicroRecall@K':>14s}  {'Precision@K':>12s}  {'NDCG@K':>8s}")
print(f"  {'-'*4}  {'-'*10}  {'-'*14}  {'-'*12}  {'-'*8}")
for k in K_VALUES:
    print(f"  {k:4d}  {sl.get(f'recall@{k}', 0):10.4f}  "
          f"{sl.get(f'micro_recall@{k}', 0):14.4f}  "
          f"{sl.get(f'precision@{k}', 0):12.4f}  "
          f"{sl.get(f'ndcg@{k}', 0):8.4f}")


In [ ]:
ma = results['macro']

print("=" * 70)
print("MACRO METRICS (per-code averaged, min_count >= {})".format(MACRO_MIN_COUNT))
print("=" * 70)
print(f"\n  {'K':>4s}  {'MacroRecall@K':>14s}  {'#Codes(R)':>10s}  {'MacroPrecision@K':>17s}  {'#Codes(P)':>10s}")
print(f"  {'-'*4}  {'-'*14}  {'-'*10}  {'-'*17}  {'-'*10}")
for k in K_VALUES:
    print(f"  {k:4d}  {ma.get(f'macro_recall@{k}', 0):14.4f}  "
          f"{ma.get(f'macro_recall@{k}_num_codes', 0):10,}  "
          f"{ma.get(f'macro_precision@{k}', 0):17.4f}  "
          f"{ma.get(f'macro_precision@{k}_num_codes', 0):10,}")


In [ ]:
print("=" * 70)
print("PRIMARY METRICS SUMMARY (K=10, K=20)")
print("=" * 70)

per_code = results['per_code']

for k in PRIMARY_K_VALUES:
    members_with_true_codes = per_code.total_samples
    members_identified = (
        per_code.overall_member_hit_count[k]
        if per_code.overall_member_hit_count is not None else 0
    )
    member_identification_rate = (
        members_identified / members_with_true_codes
        if members_with_true_codes > 0 else 0.0
    )

    print(f"\n  === K = {k} ===")
    print(f"  Micro Recall@{k}:              {sl.get(f'micro_recall@{k}', 0):.4f}")
    print(f"  Macro Recall@{k}:              {ma.get(f'macro_recall@{k}', 0):.4f}")
    print(f"  Micro Precision@{k}:           {sl.get(f'precision@{k}', 0):.4f}")
    print(f"  Macro Precision@{k}:           {ma.get(f'macro_precision@{k}', 0):.4f}")
    print(f"  NDCG@{k}:                      {sl.get(f'ndcg@{k}', 0):.4f}")
    print(f"  Members Identified@{k}:        {members_identified:,} / {members_with_true_codes:,}")
    print(f"  Member Identification Rate@{k}: {member_identification_rate:.4f}")

# Global member/sample-level identification summary for all K values.
# Note: this counts evaluated (member, day) samples, not distinct individual_id values.
global_member_rows = []
for k in K_VALUES:
    members_with_true_codes = per_code.total_samples
    members_identified = (
        per_code.overall_member_hit_count[k]
        if per_code.overall_member_hit_count is not None else 0
    )
    member_identification_rate = (
        members_identified / members_with_true_codes
        if members_with_true_codes > 0 else 0.0
    )
    streaming_recall = sl.get(f'recall@{k}', 0.0)
    global_member_rows.append({
        'K': k,
        'Members w/ True Codes': members_with_true_codes,
        'Members Identified': members_identified,
        'Member Identification Rate@K': member_identification_rate,
        'Streaming Recall@K': streaming_recall,
        'Rate - Streaming Recall': member_identification_rate - streaming_recall,
    })

global_member_df = pd.DataFrame(global_member_rows)
print("\nGlobal Member/Sample Identification Summary:")
display(global_member_df.to_string(index=False))


## 8. Code-Type-Specific Metrics

Breakdown of all metrics by code type. This reveals which clinical domains
the model predicts best and worst.


In [ ]:
ct = results['code_type']

for k in PRIMARY_K_VALUES:
    print(f"\n{'=' * 150}")
    print(f"CODE-TYPE BREAKDOWN — K = {k}")
    print(f"{'=' * 150}")

    header = (f"  {'Code Type':25s}  {'#Codes':>7s}  {'#TrueOcc':>9s}  "
              f"{'#MbrTrue':>9s}  {'#MbrHit':>8s}  {'MbrHitRate':>10s}  "
              f"{'Occ/Mbr':>8s}  {'µRecall':>8s}  {'MRecall':>8s}  "
              f"{'µPrec':>8s}  {'MPrec':>8s}")
    print(header)
    print(f"  {'-' * 143}")

    for code_type in CODE_TYPES:
        if code_type not in ct:
            continue
        m = ct[code_type]
        members_with_true = m.get('members_with_true_codes', 0)
        members_identified = m.get(f'members_identified@{k}', 0)
        member_hit_rate = (
            members_identified / members_with_true if members_with_true > 0 else 0.0
        )
        occurrences_per_member = (
            m.get('total_true_occurrences', 0) / members_with_true
            if members_with_true > 0 else 0.0
        )
        print(f"  {code_type:25s}  "
              f"{m.get('num_codes', 0):7,}  "
              f"{m.get('total_true_occurrences', 0):9,}  "
              f"{members_with_true:9,}  "
              f"{members_identified:8,}  "
              f"{member_hit_rate:10.4f}  "
              f"{occurrences_per_member:8.2f}  "
              f"{m.get(f'micro_recall@{k}', 0):8.4f}  "
              f"{m.get(f'macro_recall@{k}', 0):8.4f}  "
              f"{m.get(f'micro_precision@{k}', 0):8.4f}  "
              f"{m.get(f'macro_precision@{k}', 0):8.4f}")


In [ ]:
rows = []
for code_type in CODE_TYPES:
    if code_type not in ct:
        continue
    m = ct[code_type]
    members_with_true = m.get('members_with_true_codes', 0)
    occurrences_per_member = (
        m.get('total_true_occurrences', 0) / members_with_true
        if members_with_true > 0 else 0.0
    )
    for k in K_VALUES:
        members_identified = m.get(f'members_identified@{k}', 0)
        member_identification_rate = (
            members_identified / members_with_true if members_with_true > 0 else 0.0
        )
        rows.append({
            'Code Type': code_type,
            'K': k,
            'Num Codes': m.get('num_codes', 0),
            'Total True Occurrences': m.get('total_true_occurrences', 0),
            'Members w/ True Codes': members_with_true,
            'Members Identified': members_identified,
            'Member Identification Rate@K': member_identification_rate,
            'True Occurrences per Member': occurrences_per_member,
            'Micro Recall@K': m.get(f'micro_recall@{k}', 0),
            'Macro Recall@K': m.get(f'macro_recall@{k}', 0),
            'Micro Precision@K': m.get(f'micro_precision@{k}', 0),
            'Macro Precision@K': m.get(f'macro_precision@{k}', 0),
            'Macro Recall Num Codes': m.get(f'macro_recall@{k}_num_codes', 0),
            'Macro Precision Num Codes': m.get(f'macro_precision@{k}_num_codes', 0),
        })

metrics_df = pd.DataFrame(rows)
print("Full Code-Type Metrics DataFrame:")
display(metrics_df.to_string(index=False))

# Pivot for K=10 and K=20 comparison
for k in PRIMARY_K_VALUES:
    subset = metrics_df[metrics_df['K'] == k].copy()
    subset = subset.sort_values('Total True Occurrences', ascending=False)
    print(f"\n=== K={k} (sorted by true occurrences) ===")
    display(subset[['Code Type', 'Num Codes', 'Total True Occurrences',
                     'Members w/ True Codes', 'Members Identified',
                     'Member Identification Rate@K', 'True Occurrences per Member',
                     'Micro Recall@K', 'Macro Recall@K',
                     'Micro Precision@K', 'Macro Precision@K']].to_string(index=False))


### 8.1 Code Type Comparison with Overall Reference

Compare each code type against the **Overall** baseline across K = {1, 5, 10, 20}.
Includes member/sample-level counts:
- **Members w/ True Codes**: number of evaluated (member, day) samples that have >=1 true code of this type
- **Members Identified**: number of those samples where >=1 code of this type was correctly predicted in top-K
- **Member Identification Rate@K**: `Members Identified / Members w/ True Codes`
- **Delta vs Overall** columns: code-type metric minus the Overall baseline at the same K

In [ ]:
# ============================================================================
# BUILD comparison_df: Code Types + Overall, K = {1, 5, 10, 20}
# ============================================================================
COMPARISON_K_VALUES = (1, 5, 10, 20)

per_code = results['per_code']

def _safe_div(num, den):
    return num / den if den > 0 else 0.0

# --- Overall row from global accumulators ---
overall_rows = []
overall_by_k = {}
for k in COMPARISON_K_VALUES:
    total_true = int(per_code.code_true_count.sum())
    total_hits = int(per_code.code_hit_count[k].sum())
    total_topk = int(per_code.code_topk_count[k].sum())
    total_members = per_code.total_samples
    members_identified = (
        per_code.overall_member_hit_count[k]
        if per_code.overall_member_hit_count is not None else 0
    )

    overall_metrics = {
        'Micro Recall@K': _safe_div(total_hits, total_true),
        'Macro Recall@K': ma.get(f'macro_recall@{k}', 0.0),
        'Micro Precision@K': _safe_div(total_hits, total_topk),
        'Macro Precision@K': ma.get(f'macro_precision@{k}', 0.0),
        'Member Identification Rate@K': _safe_div(members_identified, total_members),
    }
    overall_by_k[k] = overall_metrics

    overall_rows.append({
        'Code Type': 'Overall',
        'K': k,
        'Num Codes': int((per_code.code_true_count > 0).sum()),
        'Total True Occurrences': total_true,
        'Members w/ True Codes': total_members,
        'Members Identified': members_identified,
        'True Occurrences per Member': _safe_div(total_true, total_members),
        **overall_metrics,
    })

# --- Code type rows (filter to comparison K values) ---
type_rows = []
for code_type in CODE_TYPES:
    if code_type not in ct:
        continue
    m = ct[code_type]
    members_with_true = m.get('members_with_true_codes', 0)
    total_true_occurrences = m.get('total_true_occurrences', 0)
    for k in COMPARISON_K_VALUES:
        members_identified = m.get(f'members_identified@{k}', 0)
        type_rows.append({
            'Code Type': code_type,
            'K': k,
            'Num Codes': m.get('num_codes', 0),
            'Total True Occurrences': total_true_occurrences,
            'Members w/ True Codes': members_with_true,
            'Members Identified': members_identified,
            'True Occurrences per Member': _safe_div(total_true_occurrences, members_with_true),
            'Member Identification Rate@K': _safe_div(members_identified, members_with_true),
            'Micro Recall@K': m.get(f'micro_recall@{k}', 0),
            'Macro Recall@K': m.get(f'macro_recall@{k}', 0),
            'Micro Precision@K': m.get(f'micro_precision@{k}', 0),
            'Macro Precision@K': m.get(f'macro_precision@{k}', 0),
        })

comparison_df = pd.DataFrame(overall_rows + type_rows)

# Add deltas vs Overall at each K
for metric in [
    'Micro Recall@K', 'Macro Recall@K',
    'Micro Precision@K', 'Macro Precision@K',
    'Member Identification Rate@K',
]:
    comparison_df[f'{metric} Δ vs Overall'] = comparison_df.apply(
        lambda row: row[metric] - overall_by_k[row['K']][metric], axis=1
    )

# Display per-K pivot tables
for k in COMPARISON_K_VALUES:
    subset = comparison_df[comparison_df['K'] == k].copy()
    subset = subset.set_index('Code Type')
    order = ['Overall'] + [t for t in subset.index if t != 'Overall']
    subset = subset.loc[order]

    print(f"\n{'=' * 150}")
    print(f"COMPARISON TABLE — K = {k}")
    print(f"{'=' * 150}")
    display(subset[[
        'Num Codes', 'Total True Occurrences',
        'Members w/ True Codes', 'Members Identified',
        'Member Identification Rate@K', 'Member Identification Rate@K Δ vs Overall',
        'True Occurrences per Member',
        'Micro Recall@K', 'Micro Recall@K Δ vs Overall',
        'Macro Recall@K', 'Macro Recall@K Δ vs Overall',
        'Micro Precision@K', 'Micro Precision@K Δ vs Overall',
        'Macro Precision@K', 'Macro Precision@K Δ vs Overall',
    ]])

print(f"\ncomparison_df shape: {comparison_df.shape}")
print("Columns:", list(comparison_df.columns))

### 8.2 Commercial Procedure Group Drilldown

This section narrows the intrinsic evaluation to **Commercial** validation members and reports performance for every included `prcdr_group_*` target code in the codebook.

Rationale:
- The aggregate `Procedure Groups` row can hide large differences between individual procedure families.
- Commercial members may have a different procedure distribution than the all-population validation set.
- Single-code rows keep the same metric schema used above, including member/sample identification metrics. For one code, micro and macro recall/precision are equivalent by construction, but both are retained for schema consistency.

Indexing note: `w2ind_target.ind` is stored as a 1-based target index for non-empty codes, while `conv_target(...)` converts targets to 0-based model indices with `code_val - 1`. This section therefore uses `model_target_idx = target_ind_raw - 1` for per-code accumulator lookup.

Exclusion note: procedure-group raw target id `1115` is excluded from the Commercial procedure-group evaluation, aggregate procedure summary, and per-code performance report.

In [ ]:
# ============================================================================
# 8.2.1 Review Procedure Group Codes in the Target Codebook
# ============================================================================

PROCEDURE_GROUP_PREFIX = "prcdr_group_"
COMMERCIAL_LOB_LABEL = "Commercial"
EXCLUDED_PROCEDURE_GROUP_TARGET_INDS = {1115}

procedure_group_codebook_df = (
    w2ind_target_df
    .assign(
        cd=lambda df: df["cd"].astype(str),
        target_ind_raw=lambda df: df["ind"].astype(int),
    )
    .loc[lambda df: df["cd"].str.startswith(PROCEDURE_GROUP_PREFIX), ["target_ind_raw", "cd"]]
    .sort_values("cd")
    .reset_index(drop=True)
)

# Model outputs and per-code accumulators use 0-based indices after conv_target(...).
procedure_group_codebook_df["model_target_idx"] = procedure_group_codebook_df["target_ind_raw"] - 1
procedure_group_codebook_df["procedure_group"] = procedure_group_codebook_df["cd"].str.replace(
    PROCEDURE_GROUP_PREFIX, "", regex=False
)
procedure_group_codebook_df["valid_model_index"] = procedure_group_codebook_df["model_target_idx"].between(
    0, config.target_cd_cnt - 1
)
procedure_group_codebook_df["excluded_from_commercial_eval"] = procedure_group_codebook_df[
    "target_ind_raw"
].isin(EXCLUDED_PROCEDURE_GROUP_TARGET_INDS)
procedure_group_codebook_df["include_in_commercial_eval"] = (
    procedure_group_codebook_df["valid_model_index"]
    & ~procedure_group_codebook_df["excluded_from_commercial_eval"]
)

# Corrected model-index type map for Commercial procedure-only evaluation.
# This avoids using raw w2ind_target.ind values as if they were model indices.
PROCEDURE_GROUP_CODE_TYPES = ["Procedure Group"]
procedure_group_type_to_id = {"Procedure Group": 0}
procedure_group_idx_type_array = np.full(config.target_cd_cnt, -1, dtype=np.int32)
included_procedure_model_indices = procedure_group_codebook_df.loc[
    procedure_group_codebook_df["include_in_commercial_eval"], "model_target_idx"
].astype(int).to_numpy()
procedure_group_idx_type_array[included_procedure_model_indices] = 0

print("Procedure group codebook review")
print("-" * 70)
print(f"Procedure group codes in w2ind_target: {len(procedure_group_codebook_df):,}")
print(f"Valid model-index rows: {procedure_group_codebook_df['valid_model_index'].sum():,}")
print(f"Invalid model-index rows: {(~procedure_group_codebook_df['valid_model_index']).sum():,}")
print(f"Excluded raw target ids: {sorted(EXCLUDED_PROCEDURE_GROUP_TARGET_INDS)}")
print(f"Rows excluded from Commercial evaluation: "
      f"{procedure_group_codebook_df['excluded_from_commercial_eval'].sum():,}")
print(f"Rows included in Commercial evaluation: "
      f"{procedure_group_codebook_df['include_in_commercial_eval'].sum():,}")
print(f"Raw target index range: {procedure_group_codebook_df['target_ind_raw'].min():,} to "
      f"{procedure_group_codebook_df['target_ind_raw'].max():,}")
print(f"Model target index range: {procedure_group_codebook_df['model_target_idx'].min():,} to "
      f"{procedure_group_codebook_df['model_target_idx'].max():,}")

if not procedure_group_codebook_df["valid_model_index"].all():
    print("\nWARNING: Some procedure group codebook rows do not map into model target space.")
    display(procedure_group_codebook_df.loc[~procedure_group_codebook_df["valid_model_index"]])

print("\nProcedure group codebook sample:")
display(procedure_group_codebook_df.head(50))

# Full table remains available for review/filtering in the notebook.
procedure_group_codebook_df

In [ ]:
# ============================================================================
# 8.2.2 Commercial-Only Validation Evaluation
# ============================================================================

commercial_val_df = val_df[val_df["lob"] == COMMERCIAL_LOB_LABEL].copy()

commercial_validation_provenance = {
    "lob_filter": COMMERCIAL_LOB_LABEL,
    "excluded_procedure_group_target_inds": sorted(EXCLUDED_PROCEDURE_GROUP_TARGET_INDS),
    "included_procedure_group_codes": int(procedure_group_codebook_df["include_in_commercial_eval"].sum()),
    "excluded_procedure_group_codes": int(procedure_group_codebook_df["excluded_from_commercial_eval"].sum()),
    "all_validation_members": int(len(val_df)),
    "commercial_validation_members": int(len(commercial_val_df)),
    "commercial_share_of_validation": float(len(commercial_val_df) / max(len(val_df), 1)),
    "commercial_unique_individual_ids": int(
        commercial_val_df["individual_id"].nunique()
        if "individual_id" in commercial_val_df.columns else len(commercial_val_df)
    ),
}

print("Commercial-only validation subset")
print("-" * 70)
print(f"All validation members: {len(val_df):,}")
print(f"Commercial validation members: {len(commercial_val_df):,}")
print(f"Commercial share: {commercial_validation_provenance['commercial_share_of_validation']:.4f}")
print("\nValidation LOB distribution:")
print(val_df["lob"].value_counts())

assert len(commercial_val_df) > 0, "No Commercial members found in validation set."

commercial_val_dataset = ClinicalDatasetLazy(commercial_val_df, config)
commercial_val_loader = DataLoader(
    commercial_val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=create_collate_fn(config),
    pin_memory=True,
    drop_last=False,
)

commercial_validation_provenance["commercial_val_dataset_rows"] = int(len(commercial_val_dataset))
commercial_validation_provenance["commercial_val_loader_batches"] = int(len(commercial_val_loader))

print(f"\nCommercial Validation DataLoader: {len(commercial_val_loader)} batches "
      f"({len(commercial_val_dataset)} samples, batch_size={EVAL_BATCH_SIZE})")

commercial_results = evaluate_comprehensive(
    model=model,
    dataloader=commercial_val_loader,
    config=config,
    device=device,
    k_values=K_VALUES,
    use_mixed_precision=use_mixed_precision,
    idx_type_array=procedure_group_idx_type_array,
    type_to_id=procedure_group_type_to_id,
    code_types=PROCEDURE_GROUP_CODE_TYPES,
    macro_min_count=MACRO_MIN_COUNT,
)

commercial_sl = commercial_results["sample_level"]
commercial_ma = commercial_results["macro"]
commercial_per_code = commercial_results["per_code"]
commercial_ct = commercial_results["code_type"]

commercial_validation_provenance["valid_member_day_pairs_evaluated"] = int(
    commercial_sl.get("num_samples", 0)
)
commercial_validation_provenance["commercial_eval_time_sec"] = float(
    commercial_results["eval_time_sec"]
)

print("\nCommercial-only evaluation complete")
print(f"  Valid (member, day) pairs evaluated: {commercial_sl.get('num_samples', 0):,}")
print(f"  Evaluation time: {commercial_results['eval_time_sec']:.1f}s")

In [ ]:
# ============================================================================
# 8.2.3 Per-Procedure-Group Commercial Performance
# ============================================================================

# Optional: replace None with a list like ["prcdr_group_992", "prcdr_group_800"]
# to restrict the report to specific procedure groups of interest.
PROCEDURE_GROUP_CODES_OF_INTEREST = None
PROCEDURE_GROUP_MIN_COUNT = 1

included_procedure_group_codebook_df = procedure_group_codebook_df[
    procedure_group_codebook_df["include_in_commercial_eval"]
].copy()

if PROCEDURE_GROUP_CODES_OF_INTEREST is None:
    procedure_groups_of_interest_df = included_procedure_group_codebook_df.copy()
else:
    requested_codes = set(PROCEDURE_GROUP_CODES_OF_INTEREST)
    procedure_groups_of_interest_df = included_procedure_group_codebook_df[
        included_procedure_group_codebook_df["cd"].isin(requested_codes)
    ].copy()
    missing_codes = sorted(requested_codes - set(procedure_groups_of_interest_df["cd"]))
    if missing_codes:
        print("WARNING: Requested procedure group codes were not found in the codebook:")
        print(missing_codes)


def build_single_code_performance_df(
    eval_results: Dict[str, Any],
    codebook_df: pd.DataFrame,
    k_values: Tuple[int, ...],
    min_count: int = 1,
) -> pd.DataFrame:
    """Build one row per code per K using the same metric schema as code-type reports.

    For a single code, the multi-hot target representation counts a member-day at
    most once, so Total True Occurrences and Members w/ True Codes are identical.
    Likewise, Members Identified is the single-code hit count at K.
    """
    per_code_acc = eval_results["per_code"]
    rows = []

    for _, code_row in codebook_df.iterrows():
        model_idx = int(code_row["model_target_idx"])
        valid_model_index = bool(code_row["valid_model_index"])

        if valid_model_index:
            total_true = int(per_code_acc.code_true_count[model_idx])
        else:
            total_true = 0

        for k in k_values:
            if valid_model_index:
                total_topk = int(per_code_acc.code_topk_count[k][model_idx])
                total_hits = int(per_code_acc.code_hit_count[k][model_idx])
            else:
                total_topk = 0
                total_hits = 0

            micro_recall = total_hits / total_true if total_true > 0 else 0.0
            micro_precision = total_hits / total_topk if total_topk > 0 else 0.0
            members_with_true = total_true
            members_identified = total_hits
            member_identification_rate = (
                members_identified / members_with_true if members_with_true > 0 else 0.0
            )

            rows.append({
                "Code Type": "Procedure Group",
                "Procedure Group Code": code_row["cd"],
                "Procedure Group": code_row["procedure_group"],
                "K": int(k),
                "Num Codes": 1,
                "Target Ind Raw": int(code_row["target_ind_raw"]),
                "Model Target Idx": model_idx,
                "Valid Model Index": valid_model_index,
                "Total True Occurrences": total_true,
                "Predicted in Top-K": total_topk,
                "Hit Count": total_hits,
                "Members w/ True Codes": members_with_true,
                "Members Identified": members_identified,
                "Member Identification Rate@K": member_identification_rate,
                "True Occurrences per Member": 1.0 if members_with_true > 0 else 0.0,
                "Micro Recall@K": micro_recall,
                "Macro Recall@K": micro_recall,
                "Micro Precision@K": micro_precision,
                "Macro Precision@K": micro_precision,
                "Macro Recall Num Codes": int(total_true >= min_count),
                "Macro Precision Num Codes": int(total_topk >= min_count),
                "Meets True Min Count": bool(total_true >= min_count),
                "Meets Pred Min Count": bool(total_topk >= min_count),
                "Has Commercial Support": bool(total_true > 0),
            })

    return pd.DataFrame(rows)


commercial_procedure_group_performance_df = build_single_code_performance_df(
    eval_results=commercial_results,
    codebook_df=procedure_groups_of_interest_df,
    k_values=K_VALUES,
    min_count=PROCEDURE_GROUP_MIN_COUNT,
)

print("Commercial procedure-group per-code performance")
print("-" * 70)
print(f"Procedure groups requested/reported: {procedure_groups_of_interest_df['cd'].nunique():,}")
print(f"Excluded procedure group raw target ids: {sorted(EXCLUDED_PROCEDURE_GROUP_TARGET_INDS)}")
print(f"Rows: {len(commercial_procedure_group_performance_df):,} "
      f"({procedure_groups_of_interest_df['cd'].nunique():,} codes × {len(K_VALUES)} K values)")
print(f"Procedure groups with Commercial support: "
      f"{commercial_procedure_group_performance_df.groupby('Procedure Group Code')['Has Commercial Support'].max().sum():,}")

for k in PRIMARY_K_VALUES:
    subset = commercial_procedure_group_performance_df[
        commercial_procedure_group_performance_df["K"] == k
    ].copy()
    subset = subset.sort_values(
        ["Total True Occurrences", "Micro Recall@K", "Micro Precision@K"],
        ascending=[False, False, False],
    )
    print(f"\n=== Commercial Procedure Groups — K={k} "
          "(sorted by Commercial true support) ===")
    display(subset[[
        "Procedure Group Code", "Target Ind Raw", "Model Target Idx",
        "Total True Occurrences", "Predicted in Top-K", "Hit Count",
        "Members w/ True Codes", "Members Identified",
        "Member Identification Rate@K", "True Occurrences per Member",
        "Micro Recall@K", "Macro Recall@K",
        "Micro Precision@K", "Macro Precision@K",
        "Has Commercial Support", "Meets True Min Count",
    ]].head(100).to_string(index=False))

commercial_procedure_group_performance_df

In [ ]:
# ============================================================================
# 8.2.4 Commercial Procedure Group Summary and Export
# ============================================================================

commercial_proc_summary_rows = []
commercial_proc_type_metrics = commercial_ct.get("Procedure Group", {})

for k in K_VALUES:
    members_with_true = int(commercial_proc_type_metrics.get("members_with_true_codes", 0))
    members_identified = int(commercial_proc_type_metrics.get(f"members_identified@{k}", 0))
    total_true = int(commercial_proc_type_metrics.get("total_true_occurrences", 0))

    commercial_proc_summary_rows.append({
        "Code Type": "Procedure Group Overall",
        "K": int(k),
        "Num Codes": int(commercial_proc_type_metrics.get("num_codes", 0)),
        "Total True Occurrences": total_true,
        "Members w/ True Codes": members_with_true,
        "Members Identified": members_identified,
        "Member Identification Rate@K": (
            members_identified / members_with_true if members_with_true > 0 else 0.0
        ),
        "True Occurrences per Member": (
            total_true / members_with_true if members_with_true > 0 else 0.0
        ),
        "Micro Recall@K": float(commercial_proc_type_metrics.get(f"micro_recall@{k}", 0.0)),
        "Macro Recall@K": float(commercial_proc_type_metrics.get(f"macro_recall@{k}", 0.0)),
        "Micro Precision@K": float(commercial_proc_type_metrics.get(f"micro_precision@{k}", 0.0)),
        "Macro Precision@K": float(commercial_proc_type_metrics.get(f"macro_precision@{k}", 0.0)),
        "Macro Recall Num Codes": int(
            commercial_proc_type_metrics.get(f"macro_recall@{k}_num_codes", 0)
        ),
        "Macro Precision Num Codes": int(
            commercial_proc_type_metrics.get(f"macro_precision@{k}_num_codes", 0)
        ),
    })

commercial_procedure_group_summary_df = pd.DataFrame(commercial_proc_summary_rows)

print("Commercial Procedure Group Overall Summary")
print("-" * 70)
display(commercial_procedure_group_summary_df.to_string(index=False))

commercial_proc_output_dir = Path(
    "logs/exp_round10_3lobs_formal_training/"
    "exp2b_flash_learned_pool_formal/eval_metrics/commercial_procedure_groups"
)
commercial_proc_output_dir.mkdir(parents=True, exist_ok=True)
commercial_proc_timestamp = time.strftime("%Y%m%d_%H%M%S")

commercial_proc_codebook_path = (
    commercial_proc_output_dir / f"procedure_group_codebook_{commercial_proc_timestamp}.csv"
)
commercial_proc_performance_path = (
    commercial_proc_output_dir / f"commercial_procedure_group_per_code_performance_{commercial_proc_timestamp}.csv"
)
commercial_proc_summary_path = (
    commercial_proc_output_dir / f"commercial_procedure_group_summary_{commercial_proc_timestamp}.csv"
)
commercial_proc_provenance_path = (
    commercial_proc_output_dir / f"commercial_validation_provenance_{commercial_proc_timestamp}.csv"
)

procedure_group_codebook_df.to_csv(commercial_proc_codebook_path, index=False)
commercial_procedure_group_performance_df.to_csv(commercial_proc_performance_path, index=False)
commercial_procedure_group_summary_df.to_csv(commercial_proc_summary_path, index=False)
pd.DataFrame([commercial_validation_provenance]).to_csv(
    commercial_proc_provenance_path, index=False
)

print("\nCommercial procedure group exports")
print("-" * 70)
print(f"Procedure group codebook: {commercial_proc_codebook_path}")
print(f"Per-code performance: {commercial_proc_performance_path}")
print(f"Procedure group summary: {commercial_proc_summary_path}")
print(f"Commercial validation provenance: {commercial_proc_provenance_path}")

## 9. Visualization

Visual diagnostics for both code-level performance and member/sample-level coverage:
- Micro vs macro recall/precision by code type
- Member identification rate by code type with Overall reference
- True member/sample prevalence vs identified coverage
- Heatmap across K values for member identification rate


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 6)
matplotlib.rcParams['font.size'] = 11

# ---------------------------------------------------------------------------
# 9.1 Micro vs Macro Recall/Precision by Code Type
# ---------------------------------------------------------------------------
for k in PRIMARY_K_VALUES:
    subset = metrics_df[metrics_df['K'] == k].copy()
    subset = subset.sort_values('Total True Occurrences', ascending=False)
    types = subset['Code Type'].values
    x = np.arange(len(types))
    width = 0.2

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    ax = axes[0]
    ax.bar(x - width/2, subset['Micro Recall@K'].values, width, label='Micro', color='steelblue')
    ax.bar(x + width/2, subset['Macro Recall@K'].values, width, label='Macro', color='coral')
    ax.set_xlabel('Code Type')
    ax.set_ylabel(f'Recall@{k}')
    ax.set_title(f'Recall@{k} by Code Type')
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    ax.bar(x - width/2, subset['Micro Precision@K'].values, width, label='Micro', color='steelblue')
    ax.bar(x + width/2, subset['Macro Precision@K'].values, width, label='Macro', color='coral')
    ax.set_xlabel('Code Type')
    ax.set_ylabel(f'Precision@{k}')
    ax.set_title(f'Precision@{k} by Code Type')
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)

    plt.suptitle(f'Micro vs Macro Metrics by Code Type — K={k}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------------
# 9.2 Member Identification Rate by Code Type with Overall Reference
# ---------------------------------------------------------------------------
for k in COMPARISON_K_VALUES:
    subset = comparison_df[(comparison_df['K'] == k) & (comparison_df['Code Type'] != 'Overall')].copy()
    subset = subset.sort_values('Members w/ True Codes', ascending=False)
    overall_rate = overall_by_k[k]['Member Identification Rate@K']

    fig, ax = plt.subplots(figsize=(16, 6))
    x = np.arange(len(subset))
    ax.bar(x, subset['Member Identification Rate@K'], color='seagreen')
    ax.axhline(overall_rate, color='black', linestyle='--', linewidth=1.5,
               label=f'Overall = {overall_rate:.4f}')
    ax.set_xlabel('Code Type')
    ax.set_ylabel(f'Member Identification Rate@{k}')
    ax.set_title(f'Member Identification Rate by Code Type — K={k}')
    ax.set_xticks(x)
    ax.set_xticklabels(subset['Code Type'], rotation=45, ha='right')
    ax.set_ylim(0, max(1.0, subset['Member Identification Rate@K'].max() * 1.1))
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------------
# 9.3 Members With True Codes vs Members Identified
# ---------------------------------------------------------------------------
for k in PRIMARY_K_VALUES:
    subset = comparison_df[(comparison_df['K'] == k) & (comparison_df['Code Type'] != 'Overall')].copy()
    subset = subset.sort_values('Members w/ True Codes', ascending=False)
    types = subset['Code Type'].values
    x = np.arange(len(types))
    width = 0.35

    fig, ax = plt.subplots(figsize=(18, 6))
    ax.bar(x - width/2, subset['Members w/ True Codes'], width,
           label='Members w/ True Codes', color='lightsteelblue')
    ax.bar(x + width/2, subset['Members Identified'], width,
           label='Members Identified', color='seagreen')
    ax.set_xlabel('Code Type')
    ax.set_ylabel('Evaluated (member, day) samples')
    ax.set_title(f'Member/Sample Coverage by Code Type — K={k}')
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------------
# 9.4 Heatmap: Member Identification Rate by Code Type and K
# ---------------------------------------------------------------------------
heatmap_df = (
    comparison_df[comparison_df['Code Type'] != 'Overall']
    .pivot(index='Code Type', columns='K', values='Member Identification Rate@K')
    .loc[:, list(COMPARISON_K_VALUES)]
)
heatmap_df = heatmap_df.loc[
    comparison_df[comparison_df['Code Type'] != 'Overall']
    .groupby('Code Type')['Members w/ True Codes']
    .max()
    .sort_values(ascending=False)
    .index
]

fig, ax = plt.subplots(figsize=(10, max(5, 0.55 * len(heatmap_df))))
im = ax.imshow(heatmap_df.values, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels([f'K={k}' for k in heatmap_df.columns])
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
ax.set_title('Member Identification Rate Heatmap by Code Type and K')
for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        ax.text(j, i, f'{heatmap_df.values[i, j]:.3f}', ha='center', va='center',
                color='white' if heatmap_df.values[i, j] < 0.5 else 'black')
fig.colorbar(im, ax=ax, label='Member Identification Rate')
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# 9.5 Coverage vs Performance Scatter
# ---------------------------------------------------------------------------
for k in PRIMARY_K_VALUES:
    subset = comparison_df[(comparison_df['K'] == k) & (comparison_df['Code Type'] != 'Overall')].copy()
    fig, ax = plt.subplots(figsize=(12, 7))
    sizes = np.sqrt(subset['Num Codes'].clip(lower=1)) * 20
    ax.scatter(
        subset['Members w/ True Codes'],
        subset['Member Identification Rate@K'],
        s=sizes,
        alpha=0.7,
        color='teal',
        edgecolor='black',
    )
    for _, row in subset.iterrows():
        ax.annotate(row['Code Type'],
                    (row['Members w/ True Codes'], row['Member Identification Rate@K']),
                    textcoords='offset points', xytext=(5, 4), fontsize=9)
    ax.axhline(overall_by_k[k]['Member Identification Rate@K'], color='black',
               linestyle='--', linewidth=1.2, label='Overall member identification rate')
    ax.set_xscale('log')
    ax.set_xlabel('Members w/ True Codes (log scale)')
    ax.set_ylabel(f'Member Identification Rate@{k}')
    ax.set_title(f'Coverage vs Member Identification Rate — K={k}')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
print("=" * 100)
print("MICRO-MACRO AND MEMBER COVERAGE GAP ANALYSIS")
print("=" * 100)
print("\nA large micro-macro gap indicates the model performs well on frequent codes")
print("but less consistently across rare codes within that code type.")
print("Member Rate Δ vs Overall shows whether a code type is easier/harder to identify")
print("at the member/sample level than the global baseline at the same K.\n")

for k in PRIMARY_K_VALUES:
    subset = comparison_df[(comparison_df['K'] == k) & (comparison_df['Code Type'] != 'Overall')].copy()
    subset = subset.sort_values('Total True Occurrences', ascending=False)

    print(f"  === K = {k} ===")
    print(f"  {'Code Type':25s}  {'µ-M Recall Gap':>15s}  {'µ-M Prec Gap':>13s}  "
          f"{'MbrRate Δ':>10s}  {'MbrRate':>8s}  {'#MbrTrue':>9s}  {'#True':>9s}")
    print(f"  {'-' * 104}")
    for _, row in subset.iterrows():
        recall_gap = row['Micro Recall@K'] - row['Macro Recall@K']
        prec_gap = row['Micro Precision@K'] - row['Macro Precision@K']
        print(f"  {row['Code Type']:25s}  "
              f"{recall_gap:+15.4f}  "
              f"{prec_gap:+13.4f}  "
              f"{row['Member Identification Rate@K Δ vs Overall']:+10.4f}  "
              f"{row['Member Identification Rate@K']:8.4f}  "
              f"{int(row['Members w/ True Codes']):9,}  "
              f"{int(row['Total True Occurrences']):9,}")
    print()


## 10. Save Results


In [ ]:
output_dir = Path("logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool_formal/eval_metrics")
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"comprehensive_intrinsic_metrics_{timestamp}.json"
performance_csv_path = output_dir / f"overall_and_code_type_performance_{timestamp}.csv"
provenance_csv_path = output_dir / f"validation_provenance_{timestamp}.csv"

validation_provenance_export = dict(validation_provenance) if 'validation_provenance' in globals() else {}
validation_provenance_export.update({
    'model_path': TRAINED_MODEL_PATH,
    'validation_members': int(len(val_df)),
    'validation_unique_individual_ids': int(
        val_df['individual_id'].nunique()
        if 'individual_id' in val_df.columns else len(val_df)
    ),
    'valid_member_day_pairs_evaluated': int(results['sample_level'].get('num_samples', 0)),
    'analysis_unit': 'valid (member, day) pair after flattening each member sequence over valid days',
    'eval_batch_size': int(EVAL_BATCH_SIZE),
    'num_workers': int(NUM_WORKERS),
    'macro_min_count_overall': int(MACRO_MIN_COUNT),
})
validation_provenance_df = pd.DataFrame([validation_provenance_export])

overall_rows = []
per_code = results['per_code']
for k in K_VALUES:
    total_true = int(per_code.code_true_count.sum())
    total_hits = int(per_code.code_hit_count[k].sum())
    total_topk = int(per_code.code_topk_count[k].sum())
    total_members = int(per_code.total_samples)
    members_identified = (
        int(per_code.overall_member_hit_count[k])
        if per_code.overall_member_hit_count is not None else 0
    )
    overall_rows.append({
        'Code Type': 'Overall',
        'K': int(k),
        'Num Codes': int((per_code.code_true_count > 0).sum()),
        'Total True Occurrences': total_true,
        'Members w/ True Codes': total_members,
        'Members Identified': members_identified,
        'Member Identification Rate@K': (
            members_identified / total_members if total_members > 0 else 0.0
        ),
        'True Occurrences per Member': (
            total_true / total_members if total_members > 0 else 0.0
        ),
        'Micro Recall@K': sl.get(f'micro_recall@{k}', 0.0),
        'Macro Recall@K': ma.get(f'macro_recall@{k}', 0.0),
        'Micro Precision@K': sl.get(f'precision@{k}', 0.0),
        'Macro Precision@K': ma.get(f'macro_precision@{k}', 0.0),
        'Macro Recall Num Codes': ma.get(f'macro_recall@{k}_num_codes', 0),
        'Macro Precision Num Codes': ma.get(f'macro_precision@{k}_num_codes', 0),
    })

type_rows = []
for code_type in CODE_TYPES:
    if code_type not in ct:
        continue
    metrics = ct[code_type]
    members_with_true = metrics.get('members_with_true_codes', 0)
    total_true_occurrences = metrics.get('total_true_occurrences', 0)
    true_occurrences_per_member = (
        total_true_occurrences / members_with_true if members_with_true > 0 else 0.0
    )
    for k in K_VALUES:
        members_identified = metrics.get(f'members_identified@{k}', 0)
        type_rows.append({
            'Code Type': code_type,
            'K': int(k),
            'Num Codes': int(metrics.get('num_codes', 0)),
            'Total True Occurrences': int(total_true_occurrences),
            'Members w/ True Codes': int(members_with_true),
            'Members Identified': int(members_identified),
            'Member Identification Rate@K': (
                members_identified / members_with_true if members_with_true > 0 else 0.0
            ),
            'True Occurrences per Member': true_occurrences_per_member,
            'Micro Recall@K': float(metrics.get(f'micro_recall@{k}', 0.0)),
            'Macro Recall@K': float(metrics.get(f'macro_recall@{k}', 0.0)),
            'Micro Precision@K': float(metrics.get(f'micro_precision@{k}', 0.0)),
            'Macro Precision@K': float(metrics.get(f'macro_precision@{k}', 0.0)),
            'Macro Recall Num Codes': int(metrics.get(f'macro_recall@{k}_num_codes', 0)),
            'Macro Precision Num Codes': int(metrics.get(f'macro_precision@{k}_num_codes', 0)),
        })

stakeholder_perf_df = pd.DataFrame(overall_rows + type_rows)
code_type_order = ['Overall'] + [code_type for code_type in CODE_TYPES if code_type in ct]
stakeholder_perf_df['Code Type'] = pd.Categorical(
    stakeholder_perf_df['Code Type'],
    categories=code_type_order,
    ordered=True,
    )
stakeholder_perf_df = stakeholder_perf_df.sort_values(['K', 'Code Type']).reset_index(drop=True)

stakeholder_perf_df.to_csv(performance_csv_path, index=False)
validation_provenance_df.to_csv(provenance_csv_path, index=False)

export_data = {
    'model_path': TRAINED_MODEL_PATH,
    'training_data_table': TRAINING_DATA_TABLE,
    'train_ratio': TRAIN_RATIO,
    'random_seed': RANDOM_SEED,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'k_values': list(K_VALUES),
    'macro_min_count': MACRO_MIN_COUNT,
    'num_val_members': len(val_df),
    'total_valid_day_pairs': results['sample_level'].get('num_samples', 0),
    'eval_time_sec': results['eval_time_sec'],
    'validation_provenance': validation_provenance_export,
    'csv_exports': {
        'overall_and_code_type_performance': str(performance_csv_path),
        'validation_provenance': str(provenance_csv_path),
    },
    'sample_level_metrics': {
        k: float(v) if isinstance(v, (float, np.floating))
        else int(v) if isinstance(v, (int, np.integer))
        else v
        for k, v in results['sample_level'].items()
        if isinstance(v, (int, float, np.integer, np.floating))
    },
    'macro_metrics': {
        k: float(v) if isinstance(v, (float, np.floating))
        else int(v) if isinstance(v, (int, np.integer))
        else v
        for k, v in results['macro'].items()
    },
    'code_type_metrics': results['code_type'],
    'member_level_overall': {
        f'members_identified@{k}': (
            results['per_code'].overall_member_hit_count[k]
            if results['per_code'].overall_member_hit_count is not None else 0
        )
        for k in K_VALUES
    },
}

with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"Results saved to: {output_path}")
print(f"Stakeholder performance CSV: {performance_csv_path}")
print(f"Validation provenance CSV: {provenance_csv_path}")
print(f"JSON file size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"Performance CSV rows: {len(stakeholder_perf_df):,}")